# Tokenization

First step to process text data to something than ML models can understand (it's all about the maths afterall). Tokenization just referes to splitting of text into smaller units called tokens.

In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer

In [4]:
sentences = [
    "I love machine learning",
    "Deep learning is a subset of machine learning",
    "Natural Language Processing is fun",
    "I enjoy learning new things",
    "Michelle Obama was the former First Lady of the United States",
    "Narendra Modi is the Prime Minister of India"
]

tokenizer = Tokenizer(oov_token="<OOV>") 
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index
sequences = tokenizer.texts_to_sequences(sentences)
padded_sequences = keras.preprocessing.sequence.pad_sequences(sequences, padding='post')

In [2]:
print("Word Index:\n", word_index)
print("\nSequences:\n", sequences)
print("\nPadded Sequences:\n", padded_sequences)

Word Index:
 {'<OOV>': 1, 'learning': 2, 'is': 3, 'of': 4, 'the': 5, 'i': 6, 'machine': 7, 'love': 8, 'deep': 9, 'a': 10, 'subset': 11, 'natural': 12, 'language': 13, 'processing': 14, 'fun': 15, 'enjoy': 16, 'new': 17, 'things': 18, 'michelle': 19, 'obama': 20, 'was': 21, 'former': 22, 'first': 23, 'lady': 24, 'united': 25, 'states': 26, 'narendra': 27, 'modi': 28, 'prime': 29, 'minister': 30, 'india': 31}

Sequences:
 [[6, 8, 7, 2], [9, 2, 3, 10, 11, 4, 7, 2], [12, 13, 14, 3, 15], [6, 16, 2, 17, 18], [19, 20, 21, 5, 22, 23, 24, 4, 5, 25, 26], [27, 28, 3, 5, 29, 30, 4, 31]]

Padded Sequences:
 [[ 6  8  7  2  0  0  0  0  0  0  0]
 [ 9  2  3 10 11  4  7  2  0  0  0]
 [12 13 14  3 15  0  0  0  0  0  0]
 [ 6 16  2 17 18  0  0  0  0  0  0]
 [19 20 21  5 22 23 24  4  5 25 26]
 [27 28  3  5 29 30  4 31  0  0  0]]


### Observations

- default cap for num_words is set to None, meaning all words in the corpus are considered.
- `<OOV>` token is given the index 1, padding is given the index 0.
- padding is added to the start of the sequences by default (pre-padding).

# Stemming

On large datasets, there are a lot of words that are just different forms of the same word root. Stemming is basicalling applyig a set of hard coded rules to remove affixes from words to get to the root form. Since this is hardcoded, it comes with its own set of problems. For example, the word ability would be stemmed to abil, which is not a valid word. Lets see how many of more such silly stems we can get with the Porter Stemmer. 

In [5]:
import nltk
from nltk.stem import PorterStemmer

In [7]:
stemmer = PorterStemmer()

words = ["running", "runner", "ran", "easily", "fairly", "happiness", "ability", "studies", "studying", "studied"]

for word in words:
    print(f"Original Word: {word} --> Stemmed Word: {stemmer.stem(word)}")

Original Word: running --> Stemmed Word: run
Original Word: runner --> Stemmed Word: runner
Original Word: ran --> Stemmed Word: ran
Original Word: easily --> Stemmed Word: easili
Original Word: fairly --> Stemmed Word: fairli
Original Word: happiness --> Stemmed Word: happi
Original Word: ability --> Stemmed Word: abil
Original Word: studies --> Stemmed Word: studi
Original Word: studying --> Stemmed Word: studi
Original Word: studied --> Stemmed Word: studi


### Observations

- removes ly, so easily -> easi which doesn't make sense
- removes ness, so happiness -> happi which again doesn't make sense, suffix addition is not always appended directly to the root word
- not really able to understand how it removes -ing, for runing it takes off -ing as well as an extra 'n' and for studying it takes off just the -ng along with 'y'.

Able to find this paper by C.J. van Rijsbergen, S.E. Robertson and M.F. Porter, 1980: ["New Models in Probabilistic Information Retrieval"](https://tartarus.org/martin/PorterStemmer/def.txt) which discusses the stemming algorithm in detail.

- so it follows quite a speciific set of rules to stem words: 
    - if the word ending with -ing, has its stem word containing a vowel, then it removes the -ing. Then for some reason it states that stems ending with y should have the y replaced with i. Thus studying -> study -> studi.
    - simiarly running shall have its -ing cutoff, followed by removal of double letter (except for ll, ss, zz) thus running -> runn -> run.

# Stopwords

When processing a huge dataset, it would be often beneficial to remove commonly occuring words that do not add much meaning to the "human brain". I say human brain because I think most of the data we build is according to that. To demonstrate what I mean, the sentence "Modi became Prime Minister India" is not grammartically correct, but we can still understand what it means. I think the reason why stopwords are okay to be removed is because they do not change the output to that input sentence much, because a sentence is incomplete with any of the words missing in it.

In [1]:
from datasets import load_dataset
import nltk
from nltk.corpus import stopwords
import re

# Download stopword list once; quiet to avoid extra logs
nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))

def remove_stopwords(text: str) -> str:
    tokens = re.findall(r"\b\w+\b", text.lower())
    filtered = [tok for tok in tokens if tok not in stop_words]
    return " ".join(filtered)

# Use a small slice of a news-style dataset
dataset = load_dataset("ag_news", split="train[:5]")
articles = dataset["text"]

for idx, text in enumerate(articles[:3], start=1):
    cleaned = remove_stopwords(text)
    print(f"--- Example {idx} ---")
    print("Original:        ", text)
    print("Without stopwords:", cleaned)
    print()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

--- Example 1 ---
Original:         Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
Without stopwords: wall st bears claw back black reuters reuters short sellers wall street dwindling band ultra cynics seeing green

--- Example 2 ---
Original:         Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly placed\its bets on another part of the market.
Without stopwords: carlyle looks toward commercial aerospace reuters reuters private investment firm carlyle group reputation making well timed occasionally controversial plays defense industry quietly placed bets another part market

--- Example 3 ---
Original:         Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy an

Spacy provides a lot of additional stopwords as compared to NLTK.

In [3]:
import nltk
import spacy
from nltk.corpus import stopwords

# Ensure stopwords and spaCy model are available
nltk.download("stopwords", quiet=True)
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

nltk_stop = set(stopwords.words("english"))
spacy_stop = set(nlp.Defaults.stop_words)

overlap = nltk_stop & spacy_stop
only_nltk = nltk_stop - spacy_stop
only_spacy = spacy_stop - nltk_stop

print(f"NLTK stopwords: {len(nltk_stop)}")
print(f"spaCy stopwords: {len(spacy_stop)}")
print(f"Overlap: {len(overlap)}")

NLTK stopwords: 198
spaCy stopwords: 326
Overlap: 123


# Bag of Words

Bag of Words is a simple way to represent text data in a numerical format. It creates a vocabulary of all unique words in the corpus and represents each document as a vector of word counts. Each position in the vector corresponds to a word in the vocabulary, and the value at that position indicates how many times that word appears in the document.

## Bag of words vs n-grams

Bag of Words considers individual words as features, while n-grams consider sequences of 'n' words as features.

In [2]:
from sklearn.feature_extraction.text import CountVectorizer

docs = [
    "Deep learning models love lots of data",
    "Bag of words counts unigrams",
    "N-grams capture short phrases",
    "Deep models can also use n grams"
]

# Unigram bag-of-words
cv_uni = CountVectorizer(ngram_range=(1, 1))
bow_uni = cv_uni.fit_transform(docs)
vocab_uni = cv_uni.get_feature_names_out()

# Unigram + bigram bag-of-words
cv_uni_bi = CountVectorizer(ngram_range=(1, 2))
bow_uni_bi = cv_uni_bi.fit_transform(docs)
vocab_uni_bi = cv_uni_bi.get_feature_names_out()

def top_counts(matrix, vocab, doc_idx, top_k=10):
    row = matrix.getrow(doc_idx)
    pairs = list(zip(row.indices, row.data))
    pairs.sort(key=lambda x: (-x[1], vocab[x[0]]))
    return [(vocab[i], int(c)) for i, c in pairs[:top_k]]

print("Unigram vocab (first 15):", vocab_uni[:15])
print("Doc 1 unigram counts:", top_counts(bow_uni, vocab_uni, doc_idx=0))
print()
print("Uni+bi-gram vocab (first 20):", vocab_uni_bi[:20])
print("Doc 1 uni+bi-gram counts:", top_counts(bow_uni_bi, vocab_uni_bi, doc_idx=0))

Unigram vocab (first 15): ['also' 'bag' 'can' 'capture' 'counts' 'data' 'deep' 'grams' 'learning'
 'lots' 'love' 'models' 'of' 'phrases' 'short']
Doc 1 unigram counts: [('data', 1), ('deep', 1), ('learning', 1), ('lots', 1), ('love', 1), ('models', 1), ('of', 1)]

Uni+bi-gram vocab (first 20): ['also' 'also use' 'bag' 'bag of' 'can' 'can also' 'capture'
 'capture short' 'counts' 'counts unigrams' 'data' 'deep' 'deep learning'
 'deep models' 'grams' 'grams capture' 'learning' 'learning models' 'lots'
 'lots of']
Doc 1 uni+bi-gram counts: [('data', 1), ('deep', 1), ('deep learning', 1), ('learning', 1), ('learning models', 1), ('lots', 1), ('lots of', 1), ('love', 1), ('love lots', 1), ('models', 1)]


# TF-IDF
TF-IDF (Term Frequency-Inverse Document Frequency) is a statistical measure used to evaluate the importance of a word in a document relative to a collection of documents (corpus). It combines two metrics: Term Frequency (TF) and Inverse Document Frequency (IDF). Formulae:
- TF(t) = (Number of times term t appears in a document) / (Total number of terms in the document)
- IDF(t) = log_e(Total number of documents / Number of documents with term t in it)

This value appears for each word in each document. The higher the TF-IDF score, the more important the word is to that document in the context of the entire corpus.

Got to know that sklearn uses a smoothed version of IDF calculation to prevent division by zero for words that appear in all documents. The formula used is:
- IDF(t) = log_e((1 + Total number of documents) / (1 + Number of documents with term t in it)) + 1
<br>
Also finally L2 normalization is applied to the resulting TF-IDF vectors to ensure that the length of each vector is 1. This is crucial for applying ML models so as the model doesn't get biased towards 

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer

docs = [
    "Deep learning models love lots of data",
    "Bag of words counts unigrams",
    "N-grams capture short phrases",
    "Deep learning models can also use n grams"
]

tfidf = TfidfVectorizer(ngram_range=(1, 2))
X = tfidf.fit_transform(docs)
vocab = tfidf.get_feature_names_out()

def top_tfidf(matrix, vocab, doc_idx, top_k=8):
    row = matrix.getrow(doc_idx)
    pairs = list(zip(row.indices, row.data))
    pairs.sort(key=lambda x: -x[1])
    return [(vocab[i], round(float(score), 3)) for i, score in pairs[:top_k]]

print("Vocabulary size:", len(vocab))
print("First 15 vocab items:", vocab[:15])
print()
print("Doc 1 top tf-idf:", top_tfidf(X, vocab, doc_idx=0))
print("Doc 2 top tf-idf:", top_tfidf(X, vocab, doc_idx=1))

Vocabulary size: 35
First 15 vocab items: ['also' 'also use' 'bag' 'bag of' 'can' 'can also' 'capture'
 'capture short' 'counts' 'counts unigrams' 'data' 'deep' 'deep learning'
 'grams' 'grams capture']

Doc 1 top tf-idf: [('love', 0.305), ('lots', 0.305), ('data', 0.305), ('models love', 0.305), ('love lots', 0.305), ('lots of', 0.305), ('of data', 0.305), ('deep', 0.241)]
Doc 2 top tf-idf: [('bag', 0.341), ('words', 0.341), ('counts', 0.341), ('unigrams', 0.341), ('bag of', 0.341), ('of words', 0.341), ('words counts', 0.341), ('counts unigrams', 0.341)]


# Applying TF-IDF with ML models for sentiment analysis

Pipeline followed:
1. Load dataset and split into train and test sets.
2. Preprocess text data by removing stopwords, punctuations, stemming (i believe that also converts all words to lowercase as well)
3. Convert text data into TF-IDF vectors using sklearn's TfidfVectorizer.
4. Train a Logistic Regression model on the TF-IDF vectors.
5. Evaluate the model on the test set and print standard evaluation metrics.

In [6]:
from datasets import load_dataset, concatenate_datasets
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Ensure resources
nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess(text: str) -> str:
    text = text.lower()
    tokens = re.findall(r"\b[a-z]+\b", text)
    filtered = [stemmer.stem(t) for t in tokens if t not in stop_words]
    return " ".join(filtered)

# Load IMDB splits separately
train_raw = load_dataset("imdb", split="train")
test_raw = load_dataset("imdb", split="test")

# Simple EDA: class counts in each split
train_counts = np.bincount(train_raw["label"])
test_counts = np.bincount(test_raw["label"])
print("Train label counts:", train_counts)
print("Test label counts:", test_counts)
print("Classes:", {0: "neg", 1: "pos"})

# Build balanced small samples without mixing splits
per_label_train = 1000
per_label_test = 500
train_neg = train_raw.filter(lambda e: e["label"] == 0).select(range(per_label_train))
train_pos = train_raw.filter(lambda e: e["label"] == 1).select(range(per_label_train))
test_neg = test_raw.filter(lambda e: e["label"] == 0).select(range(per_label_test))
test_pos = test_raw.filter(lambda e: e["label"] == 1).select(range(per_label_test))

train_bal = concatenate_datasets([train_neg, train_pos]).shuffle(seed=42)
test_bal = concatenate_datasets([test_neg, test_pos]).shuffle(seed=42)

X_train = list(train_bal["text"])
y_train = [int(x) for x in train_bal["label"]]
X_test = list(test_bal["text"])
y_test = [int(x) for x in test_bal["label"]]

print("Balanced train counts:", np.bincount(y_train))
print("Balanced test counts:", np.bincount(y_test))
print("Sample train doc (label {}):".format(y_train[0]), X_train[0][:200].replace("\n", " "), "...\n")

# Vectorize with preprocessing
vectorizer = TfidfVectorizer(preprocessor=preprocess, ngram_range=(1, 2), min_df=2)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)

# Evaluate
y_pred = clf.predict(X_test_vec)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.3f}")
print(classification_report(y_test, y_pred, digits=3))

Train label counts: [12500 12500]
Test label counts: [12500 12500]
Classes: {0: 'neg', 1: 'pos'}
Balanced train counts: [1000 1000]
Balanced test counts: [500 500]
Sample train doc (label 1): My only minor quibble with the film I grew up knowing as STAIRWAY TO HEAVEN, is the fact that the wonderful RAYMOND MASSEY is relegated to the last twenty or so minutes in the trial scene. And the tri ...

Accuracy: 0.828
              precision    recall  f1-score   support

           0      0.794     0.886     0.837       500
           1      0.871     0.770     0.817       500

    accuracy                          0.828      1000
   macro avg      0.832     0.828     0.827      1000
weighted avg      0.832     0.828     0.827      1000

